# BRM on the adult dataset (binary classification)

The UCI **Adult** dataset is a binary classification task: predict whether a person's income exceeds \$50K/year. This notebook runs BRM with a logistic-regression learner on data with a simulated blockwise missing pattern.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

from blockwise import BRM, simulate_blockwise_missing, datasets

In [ ]:
adult = datasets.load_adult()
print(adult.shape)
print("class balance:", adult["salary"].value_counts().to_dict())
adult.head()

## Induce blockwise missingness

In [ ]:
blocks = [
    ["age", "workclass", "education"],
    ["marital-status", "occupation", "relationship"],
]
# normalize column names across the CSV (some vendors use dot instead of hyphen)
blocks = [[c.replace("-", ".") if c.replace("-", ".") in adult.columns else c for c in b] for b in blocks]

adult_miss = simulate_blockwise_missing(adult, blocks=blocks, prop_missing=0.30, noise=0.02)
(adult_miss.isna().mean() * 100).round(1)

## Encode categoricals and split

In [ ]:
cat_cols = [c for c in adult_miss.columns if adult_miss[c].dtype.name == "category" or adult_miss[c].dtype == object]
adult_encoded = pd.get_dummies(adult_miss, columns=cat_cols, dummy_na=True, drop_first=False)

# Dummy-encoding fills the category NA slot with its own column, which hides the
# row-level missingness from BRM. We keep the original NAs in the numeric columns
# so BRM's routing still sees a meaningful missingness pattern.
num_cols = [c for c in adult_miss.columns if c not in cat_cols and c != "salary"]
for c in num_cols:
    adult_encoded[c] = adult_miss[c]

X = adult_encoded.drop(columns=["salary"])
y = adult_encoded["salary"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1234, stratify=y)

## Fit BRM with a logistic-regression learner

In [ ]:
brm_lr = BRM(estimator=LogisticRegression(max_iter=500)).fit(X_train, y_train)
print("n_blocks chosen:", brm_lr.n_blocks_)

proba = brm_lr.predict_proba(X_test)[:, 1]
preds = (proba >= 0.5).astype(int)
print(f"Accuracy: {accuracy_score(y_test, preds):.3f}")
print(f"ROC-AUC : {roc_auc_score(y_test, proba):.3f}")
print("confusion matrix:")
print(confusion_matrix(y_test, preds))

## Swap in a gradient-boosted classifier

Because BRM is learner-agnostic, the only change is the `estimator` argument.

In [ ]:
brm_gb = BRM(
    estimator=GradientBoostingClassifier(random_state=0, n_estimators=200),
    n_blocks=brm_lr.n_blocks_,
).fit(X_train, y_train)

proba_gb = brm_gb.predict_proba(X_test)[:, 1]
preds_gb = (proba_gb >= 0.5).astype(int)
print(f"GB accuracy: {accuracy_score(y_test, preds_gb):.3f}")
print(f"GB ROC-AUC : {roc_auc_score(y_test, proba_gb):.3f}")

## Citation

Srinivasan, K., Currim, F., and Ram, S. (2025). *A Reduced Modeling Approach for Making Predictions With Incomplete Data Having Blockwise Missing Patterns.* INFORMS Journal on Data Science.